In [ ]:
# CUADERNO 21 · CELDA 1. Qué trae puntajes.parquet (solo nombres de columnas y conteos)
#   Verifica que el archivo sea el de la lista congelada y muestra sus columnas y totales.
#   No muestra ningún proceso ni ningún RUC.
import duckdb, hashlib
from pathlib import Path

P = Path(r"C:\SERCOP_FINAL\modelos\20261016_1017_completo_normal_sup\puntajes.parquet")
HUELLA_CONGELADA = "eed4fbceb2cc9f3c"   # primeros 16 caracteres, de preregistro_firma.json

# 1) huella del archivo, leyéndolo por partes
h = hashlib.sha256()
with open(P, "rb") as f:
    for bloque in iter(lambda: f.read(1 << 20), b""):
        h.update(bloque)
huella = h.hexdigest()
print("Huella:", huella[:16], "->", "IGUAL a la congelada" if huella.startswith(HUELLA_CONGELADA) else "DISTINTA: para y avisa")

# 2) columnas y tipos
con = duckdb.connect()
fuente = f"read_parquet('{P.as_posix()}')"
esquema = con.execute(f"DESCRIBE SELECT * FROM {fuente}").fetchall()
print(f"\nColumnas ({len(esquema)}):")
for fila in esquema:
    print(f"  {fila[0]}: {fila[1]}")

# 3) totales
print("\nFilas:", f"{con.execute(f'SELECT count(*) FROM {fuente}').fetchone()[0]:,}".replace(",", "."))
nombres = [fila[0] for fila in esquema]
for c in ["priorizado", "top_base", "top_if", "top_ens"]:
    if c in nombres:
        n = con.execute(f"SELECT sum(CAST({c} AS INTEGER)) FROM {fuente}").fetchone()[0]
        print(f"  {c}: {n:,}".replace(",", "."))

In [ ]:
# CUADERNO 21 · CELDA 2. Lista del objetivo 6 y comparación con las listas secundarias
#   Lista principal (enmienda D, 4.1): los 3.613 procesos con priorizado = True, ordenados por el
#   puntaje del ensamble, con sus variables de mayor rareza (columna explica).
#   Listas secundarias (4.2, descriptivas): 1 % del índice simple (top_base) y de Isolation Forest
#   (top_if), comparadas con la del ensamble antes (top_ensamble) y después (priorizado) de la regla
#   de coherencia, con Jaccard total y por grupo. Solo conteos: no se cruza nada con la muestra de H3.
#   La lista se escribe SIN RUC (el ocid basta para buscar el proceso en el portal).
#   Escribe en una carpeta nueva de modelos\ y no sobrescribe nada.
import duckdb, hashlib, json, os, platform
from datetime import datetime
from pathlib import Path
import pandas as pd

P = Path(r"C:\SERCOP_FINAL\modelos\20261016_1017_completo_normal_sup\puntajes.parquet")
MODELOS = Path(r"C:\SERCOP_FINAL\modelos")
HUELLA_CONGELADA = "eed4fbceb2cc9f3c"

def sha(ruta):
    h = hashlib.sha256()
    with open(ruta, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

HUELLA = sha(P)
if not HUELLA.startswith(HUELLA_CONGELADA):
    raise SystemExit("puntajes.parquet no es el congelado: para y avisa. No se escribió nada.")

RUN = datetime.now().strftime("%Y%m%d_%H%M") + "_objetivo6"
DIR = MODELOS / RUN
DIR.mkdir(exist_ok=False)          # si ya existe, se detiene: nunca sobrescribe
LINEAS = []
def say(*a):
    t = " ".join(str(x) for x in a)
    print(t)
    LINEAS.append(t)
def miles(n):
    return f"{n:,}".replace(",", ".")

con = duckdb.connect()
F = f"read_parquet('{P.as_posix()}')"
say(f"Corrida: {RUN} | puntajes.parquet {HUELLA[:16]} (congelado) | Python {platform.python_version()} | pandas {pd.__version__}")

# 1) Lista principal: sin columnas de RUC
lista = con.execute(f"""
    SELECT ocid, grupo, familia, periodo_calc, anio_proc, monto_final,
           ensamble, base, "if" AS isolation_forest, lof, ae AS autoencoder,
           n_vars_disp, explica, top_base, top_if
    FROM {F} WHERE priorizado
""").df()
n = len(lista)
if n != 3613 or lista["ocid"].nunique() != n:
    raise SystemExit(f"La lista tiene {n} filas o ocid repetidos (se esperaban 3.613 únicos): para y avisa.")
lista = lista.sort_values(["ensamble", "ocid"], ascending=[False, True]).reset_index(drop=True)
lista.insert(0, "orden", range(1, n + 1))
lista.insert(1, "orden_en_grupo", lista.groupby("grupo").cumcount() + 1)
lista = lista.rename(columns={"top_base": "tambien_en_indice_simple", "top_if": "tambien_en_isolation_forest"})
lista.to_csv(DIR / "lista_objetivo6.csv", index=False, encoding="utf-8-sig")
lista.to_excel(DIR / "lista_objetivo6.xlsx", index=False, sheet_name="lista")
say(f"\nLista principal: {miles(n)} procesos, ordenados por el puntaje del ensamble (sin RUC).")
say(f"  Puntaje del ensamble en la lista: mínimo {lista['ensamble'].min():.4f}, mediana {lista['ensamble'].median():.4f}, máximo {lista['ensamble'].max():.4f}")
say(f"  Monto: mediana {lista['monto_final'].median():,.2f} USD; total {lista['monto_final'].sum()/1e6:,.1f} millones".replace(",", "X").replace(".", ",").replace("X", "."))
say(f"  Procesos con 'explica' vacío: {int(lista['explica'].isna().sum() + (lista['explica'].astype(str).str.strip() == '').sum())}")

# 2) Por grupo y por año
g = lista.groupby("grupo").agg(procesos=("ocid", "size"), monto_mediano=("monto_final", "median")).sort_values("procesos", ascending=False)
say("\nLista principal por grupo:")
say(g.assign(monto_mediano=g["monto_mediano"].round(2)).to_string())
a = lista.groupby("anio_proc").size().rename("procesos")
say("\nLista principal por año de inicio:")
say(a.to_string())

# 3) Concentración (solo conteos; los RUC no salen de DuckDB)
conc = con.execute(f"""
    WITH l AS (SELECT * FROM {F} WHERE priorizado),
         p AS (SELECT proveedor_ruc_final, count(*) k FROM l GROUP BY 1),
         e AS (SELECT entidad_ruc, count(*) k FROM l GROUP BY 1)
    SELECT (SELECT count(*) FROM p), (SELECT max(k) FROM p),
           (SELECT sum(k) FROM (SELECT k FROM p ORDER BY k DESC LIMIT 10)),
           (SELECT count(*) FROM e), (SELECT max(k) FROM e),
           (SELECT sum(k) FROM (SELECT k FROM e ORDER BY k DESC LIMIT 10))
""").fetchone()
say(f"\nConcentración: {miles(conc[0])} proveedores distintos (el de más procesos tiene {conc[1]}; los 10 primeros suman {conc[2]}, {conc[2]/n:.1%} de la lista)")
say(f"               {miles(conc[3])} entidades distintas (la de más procesos tiene {conc[4]}; las 10 primeras suman {conc[5]}, {conc[5]/n:.1%} de la lista)")

# 4) Variables que más aparecen en 'explica'
tok = (lista["explica"].dropna().astype(str).str.replace(r"[;|/]", ",", regex=True)
       .str.split(",").explode().str.strip())
tok = tok[tok != ""].str.replace(r"\s*[\(=:].*$", "", regex=True)
say("\nVariables que más aparecen en 'explica' (conteo de procesos que la mencionan):")
say(tok.value_counts().head(15).to_string())

# 5) Listas secundarias: solapamiento y Jaccard
pares = [("top_base", "top_ensamble"), ("top_base", "priorizado"),
         ("top_if", "top_ensamble"), ("top_if", "priorizado"), ("top_base", "top_if")]
tot, porg = [], []
for x, y in pares:
    r = con.execute(f"""SELECT sum(CAST({x} AS INT)), sum(CAST({y} AS INT)),
                               sum(CAST({x} AND {y} AS INT)), sum(CAST({x} OR {y} AS INT)) FROM {F}""").fetchone()
    tot.append(dict(lista_1=x, lista_2=y, n_1=r[0], n_2=r[1], comparten=r[2], jaccard=round(r[2] / r[3], 3)))
    d = con.execute(f"""SELECT grupo, sum(CAST({x} AS INT)) n_1, sum(CAST({y} AS INT)) n_2,
                               sum(CAST({x} AND {y} AS INT)) comparten, sum(CAST({x} OR {y} AS INT)) union_
                        FROM {F} GROUP BY grupo HAVING union_ > 0 ORDER BY grupo""").df()
    d["jaccard"] = (d["comparten"] / d["union_"]).round(3)
    d.insert(0, "par", f"{x} vs {y}")
    porg.append(d.drop(columns="union_"))
T = pd.DataFrame(tot)
G = pd.concat(porg, ignore_index=True)
T.to_csv(DIR / "comparacion_listas_total.csv", index=False, encoding="utf-8-sig")
G.to_csv(DIR / "comparacion_listas_por_grupo.csv", index=False, encoding="utf-8-sig")
say("\nListas secundarias (descriptivo): procesos que comparten y Jaccard, total")
say(T.to_string(index=False))
say("\nJaccard por grupo:")
say(G.pivot(index="grupo", columns="par", values="jaccard").to_string())

# 6) Registro
reg = dict(corrida=RUN, fecha=datetime.now().isoformat(timespec="seconds"),
           puntajes_parquet=HUELLA, enmienda="D, secciones 4.1 y 4.2",
           procesos=n, archivos={f.name: sha(f) for f in sorted(DIR.iterdir()) if f.is_file()},
           python=platform.python_version(), pandas=pd.__version__, duckdb=duckdb.__version__)
(DIR / "registro_objetivo6.json").write_text(json.dumps(reg, ensure_ascii=False, indent=1), encoding="utf-8")
say(f"\nHuella de lista_objetivo6.csv: {reg['archivos']['lista_objetivo6.csv']}")
say("Archivos en", DIR)
(DIR / "resumen_para_chat.txt").write_text("\n".join(LINEAS), encoding="utf-8")
print("Pega en el chat esta salida (o resumen_para_chat.txt). No trae RUC ni procesos uno por uno.")